# Execution Algorithms

Compare deterministic TWAP, VWAP, POV, and implementation-shortfall schedules for one parent order.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
from lob_engine.core.orders import Side
from lob_engine.execution import ImplementationShortfallExecutor, POVExecutor, ParentOrder, TWAPExecutor, VWAPExecutor

parent = ParentOrder('P-NB', Side.BUY, 5000, 0, 60, arrival_price=100.0)
algorithms = [
    TWAPExecutor(parent, slices=10),
    VWAPExecutor(parent, volume_curve=[0.08, 0.09, 0.1, 0.12, 0.14, 0.14, 0.12, 0.1, 0.07, 0.04]),
    POVExecutor(parent, market_volumes=[10000, 12000, 9000, 11000, 10500], participation_rate=0.1),
    ImplementationShortfallExecutor(parent, slices=10, urgency=0.7),
]
schedules = pd.concat([algo.build_schedule() for algo in algorithms], ignore_index=True)
schedules.head()

In [ ]:
fig = px.bar(schedules, x='timestamp', y='quantity', color='algorithm', barmode='group', title='Child Order Schedules')
fig.show()

In [ ]:
prices = [100.00, 100.01, 100.02, 100.00, 99.99, 100.03, 100.04, 100.02, 100.01, 100.00]
rows = []
for algo in algorithms:
    result = algo.execute_against_prices(prices)
    rows.append({'algorithm': algo.name, **result.metrics})
metrics = pd.DataFrame(rows)
metrics[['algorithm', 'average_fill_price', 'slippage_bps', 'implementation_shortfall_bps', 'fill_ratio']]